In [ ]:
import glob, os
import ast
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from astropy.io import fits
from scipy.optimize import curve_fit
import data_manage_lib as dml

%matplotlib widget

In [ ]:
       
def get_chi2(data, model, error):
    chi2 = np.sum((data - model)**2 / error**2)
    return chi2

def cos4(alpha, thetaE=0, amp=1, offset=0):
    return amp *(np.cos(np.radians(thetaE - alpha)))**4 + offset

def cos2cos2(alpha, delta_theta, amp=1, offset=0):
    return amp * np.cos(np.radians(alpha))**2 * (np.cos(np.radians(alpha + 90 - delta_theta))**2 + offset)


# Scans automatisés

Les fichiers sont nommés Scan_Date_Time.fits

In [ ]:
# plot_path = '/home/lmousset/Projets_Recherche/COSMOCal/Tests_IAS_2026/Plots/'
plot_path = "C:\\Users\\Administrator\\Documents\\Scripts_Commande_VNA\\Louise_dev\\Plots\\"
save_plots = True
#data_path = '/home/lmousset/Projets_Recherche/COSMOCal/Tests_IAS_2026/Data/'
data_path = "C:\\Users\\Administrator\\Documents\\Scripts_Commande_VNA\\CosmoCal_data\\"
scan = '20260312_165051'

hdul = fits.open(data_path + f"Scan_{scan}.fits")
hdul.info()  # View structure

### Get informations from the fits header

In [ ]:
header = hdul[0].header
header

In [ ]:
### For the first scans, alpha (the polariser angle) was called theta
# alpha_min = header['THETA_MI']
# alpha_max = header['THETA_MA']

alpha_min = header['ALPH_MI']
alpha_max = header['ALPH_MA']
step = header['STEP']
nfreq = header['POINTS']
traces = ['S21', 'S12', 'S11', 'S22']

alpha = np.arange(alpha_min, alpha_max + step, step)
print(alpha.shape)

# For the first scans, THETA_R is not in the header but it was set to 0.0 during the acquisition, so we can use that as default value
if dml.has_key(hdul, key='THETA_R'):
    thetaR = header['THETA_R']
else:
    thetaR = 0.0
print(thetaR)

### Get the data from the fits

In [ ]:
freq_samples = hdul[0].data *1e-9  # Access frequency samples [GHz]
mag = hdul[1].data  # Access magnitude [dB]
phi = hdul[2].data  # Access phase [deg]

fstart_ind = np.argwhere(freq_samples > 127)[0][0]
print(f"First frequency sample above 127 GHz is at index {fstart_ind}, corresponding to {freq_samples[fstart_ind]:.2f} GHz")

### For the first scans, magnitude and phase were in the first two extensions, the frequency samples were not saved
# mag = hdul[0].data  # Access magnitude [dB]
# phi = hdul[1].data  # Access phase [deg]

print(mag.shape)  # Check shape of magnitude data [(nstep, nS, points)]
nstep, nS, nfreqs = mag.shape
print(f"Number of steps: {nstep}, number of S-parameters: {nS}, number of frequency points: {nfreqs}")

### Convert magnitude from dB to linear scale
mag_lin = 10**(mag/10)
print(freq_samples)

## Plot the data

### Plot en fonction de l'angle du polariseur $\alpha$

In [ ]:
### Plot plusieurs fréquences
#freqs = [0,2, 3, 10, 15, 20, 30, 40]

freqs = np.arange(fstart_ind, nfreqs)

nf = len(freqs)
tt = np.arange(0, 380)

# Utiliser les vraies valeurs de fréquence pour la normalisation
freq_values = freq_samples[freqs]
norm = Normalize(vmin=freq_values.min(), vmax=freq_values.max())
cmap = plt.cm.get_cmap('viridis')

fig, ax = plt.subplots(2, 2, figsize=(12, 9))
ax = ax.ravel()
for i, f in enumerate(freqs):
    color = cmap(norm(freq_samples[f]))
    ax[0].plot(alpha, mag_lin[:, 0, f], '.-', color=color) # S21
    ax[1].plot(alpha, mag_lin[:, 1, f], '.-', color=color) # S12
    ax[2].plot(alpha, mag_lin[:, 2, f], '.-', color=color) # S11
    ax[3].plot(alpha, mag_lin[:, 3, f], '.-', color=color) # S22

ax[0].set_xlabel("alpha (°)")
ax[0].set_ylabel("S21")
#ax[0].set_yscale('log')
ax[0].grid(10)

ax[1].set_xlabel("alpha (°)")
ax[1].set_ylabel("S12")
#ax[1].set_yscale('log')
ax[1].grid(10)

ax[2].set_xlabel("alpha (°)")
ax[2].set_ylabel("S11")
#ax[2].set_yscale('log')
ax[2].grid(10)

ax[3].set_xlabel("alpha (°)")
ax[3].set_ylabel("S22")
#ax[3].set_yscale('log')
ax[3].grid(10)

# créer une seule colorbar pour tous les plots
sm = ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])  # nécessaire pour colorbar
cbar_ax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
cbar = fig.colorbar(sm, cax=cbar_ax)
cbar.set_label('Fréquence [GHz]')

fig.tight_layout(rect=[0, 0, 0.9, 1])
if save_plots:
    fig.savefig(plot_path + f"Scan_{scan}_allfrequencies.png", dpi=300)

In [ ]:
# Plot la moyenne sur les fréquences

mean_mag_lin = np.mean(mag_lin[:, :, fstart_ind:], axis=2)

tt = np.arange(0, 380)
fig, ax = plt.subplots(2, 2, figsize=(8, 8))
ax = ax.ravel()

xx = np.arange(0, 380)
#plt.plot(xx, np.cos(np.radians(xx))**4, label=('cos^4'), color='red')



ax[0].plot(alpha, mean_mag_lin[:, 0], 'o-', color='b')
#ax[0].plot(xx, np.cos(np.radians(xx))**2 * np.cos(np.radians(xx+88))**2*3.5, label=('cos^2*cos^2'), color='orange')
ax[0].plot(tt, cos4(tt, 3) * np.max(mean_mag_lin[:, 0]), 'orange')
ax[0].set_xlabel("alpha (°)")
ax[0].set_ylabel("S21")
#ax[0].set_yscale('log')
ax[0].grid(10)

ax[1].plot(alpha, mean_mag_lin[:, 1], 'o-', color= 'b')
ax[1].plot(tt, cos4(tt, 3)*np.max(mean_mag_lin[:, 1]), 'orange')
ax[1].set_xlabel("alpha (°)")
ax[1].set_ylabel("S12")
#ax[1].set_yscale('log')
ax[1].grid(10)

ax[2].plot(alpha, mean_mag_lin[:, 2], 'o-', color= 'b')
ax[2].plot(tt, (cos4(tt, 3+90)* (np.max(mean_mag_lin[:, 2]) - np.min(mean_mag_lin[:, 2])))+ np.min(mean_mag_lin[:, 2]), 'orange')
ax[2].set_xlabel("alpha (°)")
ax[2].set_ylabel("S11")
#ax[2].set_yscale('log')
ax[2].grid(10)

ax[3].plot(alpha, mean_mag_lin[:, 3], 'o-', color= 'b')
ax[3].plot(tt, (cos4(tt, 3+90)* (np.max(mean_mag_lin[:, 3]) - np.min(mean_mag_lin[:, 3])))+ np.min(mean_mag_lin[:, 3]), 'orange')
ax[3].set_xlabel("alpha (°)")
ax[3].set_ylabel("S22")
#ax[3].set_yscale('log')
ax[3].grid(10)

fig.tight_layout()
if save_plots:
    fig.savefig(plot_path + f"Scan_{scan}_meanfrequencies.png", dpi=300)


## Plot en fonction de la fréquence

In [ ]:
# Normalisation pour chaque angle par la moyenne sur les fréquences
ang = np.arange(39)
mag_lin_norm = np.zeros_like(mag_lin)


for i, a in enumerate(ang):
    mag_lin_norm[i, :, :] = mag_lin[a, :, :]/np.mean(mag_lin[a, :, :])
print(mag_lin_norm.shape)

In [ ]:
nang = 39
tt = np.arange(0, 380)
colors, norm, cmap = dml.get_colors(nang, cmap_name='plasma', vmin=0, vmax=nang-1)

fig, ax = plt.subplots(2, 2, figsize=(10, 10))
ax = ax.ravel()
for i, a in enumerate(np.arange(0, nang)):
    ax[0].plot(freq_samples, mag_lin[a, 0, :], '.-', color=colors[i]) # S21
    ax[1].plot(freq_samples, mag_lin[a, 1, :], '.-', color=colors[i]) # S12
    ax[2].plot(freq_samples, mag_lin[a, 2, :], '.-', color=colors[i]) # S11
    ax[3].plot(freq_samples, mag_lin[a, 3, :], '.-', color=colors[i]) # S22

ax[0].set_xlabel("Frequency [GHz]")
ax[0].set_ylabel("S21")
#ax[0].set_yscale('log')
ax[0].grid(10)

ax[1].set_xlabel("Frequency [GHz]")
ax[1].set_ylabel("S12")
#ax[1].set_yscale('log')
ax[1].grid(10)

ax[2].set_xlabel("Frequency [GHz]")
ax[2].set_ylabel("S11")
#ax[2].set_yscale('log')
ax[2].grid(10)

ax[3].set_xlabel("Frequency [GHz]")
ax[3].set_ylabel("S22")
#ax[3].set_yscale('log')
ax[3].grid(10)

#créer une colorbar qui montre l'échelle d'indices (ou de valeurs réelles)
sm = ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])  # nécessaire pour colorbar
cbar = plt.colorbar(sm, ax=ax[0])
cbar = plt.colorbar(sm, ax=ax[1])
cbar = plt.colorbar(sm, ax=ax[2])
cbar = plt.colorbar(sm, ax=ax[3])
cbar.set_label('Indice de langle')

fig.tight_layout()
if save_plots:
    fig.savefig(plot_path + f"Scan_{scan}_allangles.png", dpi=300)

In [ ]:
# Moyenne sur les angles
freqs = np.arange(0, 60)
print(freqs)
ang = [1] # indice de l'angle
ang = np.arange(39)
nf = len(ang)
tt = np.arange(0, 380)

fig, ax = plt.subplots(2, 2, figsize=(10, 10))
ax = ax.ravel()


ax[0].plot(np.mean(mag_lin_norm[:, 0, :], axis=0), color='black', linestyle='--')
ax[1].plot(np.mean(mag_lin_norm[:, 1, :], axis=0), color='black', linestyle='--')
ax[2].plot(np.mean(mag_lin_norm[:, 2, :], axis=0), color='black', linestyle='--')
ax[3].plot(np.mean(mag_lin_norm[:, 3, :], axis=0), color='black', linestyle='--')
# ax[1].plot(np.mean(mag_lin[:, 1, :], axis=0), color='black', linestyle='--')
# ax[1].axhline(np.mean(mag_lin[0, 1, :]), color='black', linestyle='--')

ax[0].set_xlabel("Frequency [GHz]")
ax[0].set_ylabel("S21")
#ax[0].set_yscale('log')
ax[0].grid(10)

ax[1].set_xlabel("Frequency [GHz]")
ax[1].set_ylabel("S12")
#ax[1].set_yscale('log')
ax[1].grid(10)


ax[2].set_xlabel("Frequency [GHz]")
ax[2].set_ylabel("S11")
#ax[2].set_yscale('log')
ax[2].grid(10)


ax[3].set_xlabel("Frequency [GHz]")
ax[3].set_ylabel("S22")
#ax[3].set_yscale('log')
ax[3].grid(10)


fig.tight_layout()

# Ajustement des S-parameters en fonction de l'angle $\alpha$

Le modèle ajusté est
$$
M = A \times \cos^4(\theta_E - \alpha) + B
$$
On ajuste 3 paramètres : l'amplitude $A$, l'offset $B$ (non nul pour S11 et S22) et l'angle de polarisation de cosmocal $\theta_E$.

Pour l'instant, on ne connait pas les barres d'erreur.

### Ajustement de la moyenne sur les fréquences

In [ ]:
# def cos4(alpha, thetaE=0, amp=1, offset=0):
#     return amp *(np.cos(np.radians(thetaE - alpha)))**4 + offset 

def cos4(alpha, thetaE=0, amp=1):
    return amp *(np.cos(np.radians(thetaE - alpha)))**4

In [ ]:
# Take the mean
t = 0 # select the S parameter
if t == 0 or t==1:
    thetaE0 = 0
else :
    thetaE0 = 90

signal = mean_mag_lin[:, t]

# Normalize to the max
#signal /= np.max(signal)

# Fit the data
popt, pcov = curve_fit(cos4, alpha, signal, p0=[thetaE0, 1])
error = np.sqrt(np.diag(pcov))

print("Paramètres optimisés :", popt)
print("Erreurs sur les paramètres :", error)

# Residus et chi2 à partir des résidus sur le fit (just a check)
residuals = signal - cos4(alpha, *popt)
std_res = np.std(residuals)
print("Écart-type des résidus :", std_res)

chi2 = get_chi2(signal, cos4(alpha, *popt), std_res)
print("Chi2 (calculé avec le STD sur les résidus du fit) :", chi2)

chi2_red = chi2 / (len(alpha) - len(popt))
print("Chi2 réduit :", chi2_red)

if t == 0 or t==1:
    thetaE = popt[0]
else :
    thetaE = popt[0] - 90

fig = plt.figure()
plt.errorbar(alpha, signal, yerr=None, marker='.', ls='None')
plt.plot(tt, cos4(tt, *popt), 'orange', label='Fit cos^4')
plt.xlabel("alpha (°)")
plt.ylabel(f"{traces[t]}")
plt.title(r'$\theta_E$ = {:.3f} ± {:.3f}°'.format(thetaE, error[0]))
plt.grid(10)
plt.tight_layout()
if save_plots:
    fig.savefig(plot_path + f"Scan_{scan}_Fitmean_{traces[t]}.png", dpi=300)

### Ajustement par fréquence

In [ ]:
# Loop over frequencies
t = 0# trace index
if t == 0 or t==1:
    thetaE0 = 0
else :
    thetaE0 = 90

allthetaE = []
alloffset = []
allthetaE_err = []
allchi2_red = []
fig, axs = plt.subplots(1, 2, figsize=(11, 5))
norm = Normalize(vmin=freq_values.min(), vmax=freq_values.max())
cmap = plt.cm.get_cmap('viridis')

for f in range(fstart_ind, nfreqs):
    signal = mag_lin[:, t, f]

    # Fit the data
    popt, pcov = curve_fit(cos4, alpha, signal, p0=[thetaE0, 1])
    error = np.sqrt(np.diag(pcov))
    if t == 0 or t==1:
        thetaE = popt[0]
    else :
        thetaE = popt[0] - 90
    allthetaE.append(thetaE)
    allthetaE_err.append(error[0])
    #alloffset.append(popt[2])

    # Residus et chi2
    residuals = signal - cos4(alpha, *popt)
    std_res = np.std(residuals)
    chi2 = get_chi2(signal, cos4(alpha, *popt), std_res)

    chi2_red = chi2 / (len(alpha) - len(popt))
    allchi2_red.append(chi2_red)
    
    if f in range(100, 242, 5):
        color = cmap(norm(freq_samples[f]))
        axs[0].errorbar(alpha, signal, yerr=None, marker='.', ls='None', label=f"{freq_samples[f]:.1f}", color=color)
        axs[0].plot(tt, cos4(tt, *popt), 'k', color='grey')

        axs[1].errorbar(alpha, residuals, yerr=None, marker='.', label=f"{freq_samples[f]:.1f}", color=color)

axs[0].set_xlabel("alpha (°)")
axs[0].set_ylabel(f"{traces[t]}")
axs[0].set_title(r"Fit per frequency")
axs[0].grid(10)

axs[1].set_xlabel("alpha (°)")
axs[1].set_ylabel("Residuals")
axs[1].grid(10)

# Une seule légende pour tous les subplots
handles, labels = axs[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="center left",
            bbox_to_anchor=(0.80, 0.5), title="Frequency [GHz]", fontsize=7)

fig.tight_layout(rect=[0, 0, 0.82, 1])

if save_plots:
    fig.savefig(plot_path + f"Scan_{scan}_Fiteachfreq_{traces[t]}.png", dpi=300)

allthetaE = np.array(allthetaE)
allthetaE_err = np.array(allthetaE_err)
#alloffset = np.array(alloffset)
allchi2_red = np.array(allchi2_red)
print(allchi2_red)

#print(alloffset)

In [ ]:
# Erreur sur la moyenne de thetaE
erreur = np.sqrt(np.sum(allthetaE_err**2 )) / nfreq
print(erreur)

mthetaE = np.mean(allthetaE)
print("ThetaE moyen :", mthetaE)

stdthetaE = np.std(allthetaE) #/ np.sqrt(nfreq)
print("Écart-type de thetaE :", stdthetaE)

fig = plt.figure()
plt.errorbar(freq_samples[fstart_ind:], allthetaE, yerr=allthetaE_err, fmt='.')
plt.axhline(mthetaE, color='orange', label=r'$Mean$')
plt.axhline(mthetaE-stdthetaE, color='orange', ls='--')
plt.axhline(mthetaE+stdthetaE, color='orange', label=r'$Mean \pm STD$', ls='--')
plt.xlabel("Fréquence [GHz]")
plt.ylabel(r'$\theta_E$ fit (°)')
plt.title(rf"{traces[t]} -- $\langle\theta_E\rangle$ = {mthetaE:.3f} ± {erreur:.3f}°")
# plt.ylim(-5, 5)
plt.grid(10)
plt.legend()
if save_plots:
    fig.savefig(plot_path + f"Scan_{scan}_ThetaE_vs_freq_{traces[t]}.png", dpi=300)

## Estimation d'une erreur

Le 20/02/2026, on a fait N=5 scans identiques sans rien changer. Cela va nous permettre d'estimer une erreur.

Notre tableau de données a 4 dimensions : [i, t, a, f]
avec
* i l'indice du scan, on a fait N=5 scans
* t l'indice du coeff de scattering (0 pour S21 et 1 pour S12)
* a pour l'angle alpha du polariseur, on a 39 positions entre 0 et 380°
* f pour la fréquence, on a 60 fréquences

Nous allons calculer le STD sur les N scans.

In [ ]:
# Get the data
mag = []

os.chdir(path_save)

i = 0
for file in glob.glob("Scan_20260220_*.fits"):
    print(file)
    hdul = fits.open(path_save + file)
    header = hdul[0].header
    if dml.has_key(hdul, key='THETA_R'):
        thetaR = header['THETA_R']
    else:
        thetaR = 0.0
    if thetaR == 0.0: # Get only the scan with THETA_R=0
        print(i, file)
        mag.append(hdul[0].data)  # Access magnitude
    i+=1

mag = np.array(mag)
print(mag.shape)

mag_lin = 10**(mag / 10)

nscans, nstep, nS, nfreq = mag_lin.shape

In [ ]:
mean_over_scans = np.mean(mag_lin, axis=0)
std_over_scans = np.std(mag_lin, axis=0)
print(std_over_scans.shape)

In [ ]:
f = 10

fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax = ax.ravel()

### S21
ax[0].errorbar(alpha, mean_over_scans[:, 0, f], yerr=std_over_scans[:, 0, f], color='b', ls='None', marker='.')
ax[0].set_xlabel("alpha (°)")
ax[0].set_ylabel("Mean over scans")
ax[0].set_title(f"S21 - Frequency index: {f}")
#ax[0].set_yscale('log')
ax[0].grid(10)

### S12
ax[1].errorbar(alpha, mean_over_scans[:, 1, f], yerr=std_over_scans[:, 1, f], color='b', ls='None', marker='.')
ax[1].set_xlabel("alpha (°)")
ax[1].set_ylabel("Mean over scans")
ax[1].set_title(f"S12 - Frequency index: {f}")
#ax[1].set_yscale('log')
ax[1].grid(10)

fig.tight_layout()
#fig.savefig(f"../Plots/Scan_MEAN_over_scans.png", dpi=300)


In [ ]:
f = 10

fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax = ax.ravel()

### S21
ax[0].plot(alpha, std_over_scans[:, 0, f], '.', color='b')
ax[0].set_xlabel("alpha (°)")
ax[0].set_ylabel("STD over scans")
ax[0].set_title(f"S21 - Frequency index: {f}")
#ax[0].set_yscale('log')
ax[0].grid(10)

### S12
ax[1].plot(alpha, std_over_scans[:, 1, f], '.', color='b')
ax[1].set_xlabel("alpha (°)")
ax[1].set_ylabel("STD over scans")
ax[1].set_title(f"S12 - Frequency index: {f}")
#ax[1].set_yscale('log')
ax[1].grid(10)

fig.tight_layout()
#fig.savefig(f"../Plots/Scan_STD_over_scans.png", dpi=300)

In [ ]:
t = 0
plt.figure()
for f in range(10):
    plt.plot(mean_over_scans[:, t, f], std_over_scans[:, t, f]**2, 'o')

In [ ]:
mean_over_freq = np.mean(mag_lin, axis=3)
std_over_freq = np.std(mag_lin, axis=3)
print(std_over_freq.shape)


In [ ]:
i = 0 # Scan

fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax = ax.ravel()

### S21
ax[0].errorbar(alpha, mean_over_freq[i, :, 0], yerr=std_over_freq[i, :, 0], color='r', ls='None', marker='.')
ax[0].set_xlabel("alpha (°)")
ax[0].set_ylabel("Mean over freq")
ax[0].set_title(f"S21 - Scan index: {i}")
#ax[0].set_yscale('log')
ax[0].grid(10)

### S12
ax[1].errorbar(alpha, mean_over_freq[i, :, 1], yerr=std_over_freq[i, :, 0], color='r', ls='None', marker='.')
ax[1].set_xlabel("alpha (°)")
ax[1].set_ylabel("Mean over freq")
ax[1].set_title(f"S12 - Scan index: {i}")
#ax[1].set_yscale('log')
ax[1].grid(10)

fig.tight_layout()
#fig.savefig(f"../Plots/Scan_MEAN_over_freqs.png", dpi=300)

In [ ]:
i = 0 # Scan

fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax = ax.ravel()

### S21
ax[0].plot(alpha, std_over_freq[i, :, 0], color='r', ls='None', marker='.')
ax[0].set_xlabel("alpha (°)")
ax[0].set_ylabel("STD over freq")
ax[0].set_title(f"S21 - Scan index: {i}")
#ax[0].set_yscale('log')
ax[0].grid(10)

### S12
ax[1].plot(alpha, std_over_freq[i, :, 0], color='r', ls='None', marker='.')
ax[1].set_xlabel("alpha (°)")
ax[1].set_ylabel("STD over freq")
ax[1].set_title(f"S12 - Scan index: {i}")
#ax[1].set_yscale('log')
ax[1].grid(10)

fig.tight_layout()
#fig.savefig(f"../Plots/Scan_STD_over_freqs.png", dpi=300)

In [ ]:
t = 0
fig = plt.figure()
for s in range(nscans):
    plt.plot(mean_over_freq[s, :, t], std_over_freq[s, :, t], 'o', label=f'Scan {s}')
plt.xlabel('Mean')
plt.ylabel('STD')
plt.legend()
#fig.savefig(f"../Plots/Scan_STDvsMEAN.png", dpi=300)


## Ajustement de la moyenne sur les scans par fréquence avec des barres d'erreur

On considère que les scans sont indépendants. Donc on peut diviser le STD par sqrt(N) pour avoir l'erreur sur la moyenne.

In [ ]:
f = 12 # Frequency index 
t = 0 # trace index (S21 or S12)

signal = mean_over_scans[:, t, f]
sigma = std_over_scans[:, t, f] / np.sqrt(nscans)  # Erreur sur la moyenne

# Normalize to the max
#signal /= np.max(signal)

# Fit the data
popt, pcov = curve_fit(cos4, alpha, signal, sigma=sigma, absolute_sigma=True)
error = np.sqrt(np.diag(pcov))

print("Paramètres optimisés :", popt)
print("Erreurs sur les paramètres :", error)

# Residus et chi2
residuals = signal - cos4(alpha, *popt)
std_res = np.std(residuals)
print("Écart-type des résidus :", std_res)
chi2 = get_chi2(signal, cos4(alpha, *popt), sigma)
print("Chi2 :", chi2)

chi2_red = chi2 / (len(alpha) - len(popt))
print("Chi2 réduit :", chi2_red)

fig = plt.figure()
plt.errorbar(alpha, signal, yerr=sigma, marker='.', ls='None')
plt.plot(tt, cos4(tt, *popt), 'orange', label='Fit cos^4')
plt.xlabel("alpha (°)")
plt.ylabel("S21 mean over scans")
plt.title(r'$\theta_E$ = {:.3f} ± {:.3f}°'.format(popt[0], error[0]))
plt.grid(10)
#fig.savefig(f"../Plots/Scan_Fit_freq{f}.png", dpi=300)

In [ ]:
# Loop over frequencies
t = 0 # trace index (S21 or S12)

allthetaE = []
allthetaE_err = []
allchi2_red = []

for f in range(60):
    signal = mean_over_scans[:, t, f]
    sigma = std_over_scans[:, t, f] / np.sqrt(nscans)  # Erreur sur la moyenne

    # Normalize to the max
    #signal /= np.max(signal)

    # Fit the data
    popt, pcov = curve_fit(cos4, alpha, signal, sigma=sigma, absolute_sigma=True)
    error = np.sqrt(np.diag(pcov))

    print("Paramètres optimisés :", popt)
    allthetaE.append(popt[0])
    print("Erreurs sur les paramètres :", error)
    allthetaE_err.append(error[0])

    # Residus et chi2
    residuals = signal - cos4(alpha, *popt)
    std_res = np.std(residuals)
    print("Écart-type des résidus :", std_res)
    chi2 = get_chi2(signal, cos4(alpha, *popt), sigma)
    print("Chi2 :", chi2)

    chi2_red = chi2 / (len(alpha) - len(popt))
    allchi2_red.append(chi2_red)
    print("Chi2 réduit :", chi2_red)

In [ ]:
print(np.array(np.round(allchi2_red)))
allthetaE = np.array(allthetaE)
allthetaE_err = np.array(allthetaE_err)

# Erreur sur la moyenne de thetaE
erreur = np.sqrt(np.sum(allthetaE_err**2 )) / nfreq
print(erreur)

mthetaE = np.mean(allthetaE)
print("ThetaE moyen :", mthetaE)
stdthetaE = np.std(allthetaE) / np.sqrt(nfreq)
print("Écart-type de thetaE :", stdthetaE)

fig = plt.figure()
plt.errorbar(np.arange(len(allthetaE)), allthetaE, yerr=allthetaE_err, fmt='.')
plt.axhline(mthetaE, color='orange', label=r'$Mean$')
plt.axhline(mthetaE-stdthetaE, color='orange', ls='--')
plt.axhline(mthetaE+stdthetaE, color='orange', label=r'$Mean \pm STD/\sqrt{N}$', ls='--')
plt.xlabel("Fréquence index")
plt.ylabel(r'$\theta_E$ (°)')
plt.grid(10)
plt.legend()
#fig.savefig(f"../Plots/Scan_Fit_allthetaE", dpi=300)

# Ajustement de la moyenne

On ajuste la moyenne sur les fréquences et sur les scans.

In [ ]:
mean_over_freqscans = np.mean(mag_lin, axis=(0, 3))
std_over_freqscans = np.std(mag_lin, axis=(0, 3))
print(mean_over_freqscans.shape)

# pour 1 seul scan
#mean_over_freqscans = np.mean(mag_lin, axis=-1)
#std_over_freqscans = np.std(mag_lin, axis=-1)
#print(mean_over_freqscans.shape)

In [ ]:
# Take the mean
t = 0
signal = mean_over_freqscans[:, t]
sigma = std_over_freqscans[:, t] / (np.sqrt(nscans * nfreq))

# Normalize to the max
#signal /= np.max(signal)

# Fit the data
popt, pcov = curve_fit(cos4, alpha, signal, sigma=sigma, absolute_sigma=True)
error = np.sqrt(np.diag(pcov))

print("Paramètres optimisés :", popt)
print("Erreurs sur les paramètres :", error)

# Residus et chi2
residuals = signal - cos4(alpha, *popt)
std_res = np.std(residuals)
print("Écart-type des résidus :", std_res)

chi2 = get_chi2(signal, cos4(alpha, *popt), sigma)
print("Chi2 :", chi2)

chi2_red = chi2 / (len(alpha) - len(popt))
print("Chi2 réduit :", chi2_red)

fig = plt.figure()
plt.errorbar(alpha, signal, yerr=sigma, marker='.', ls='None')
plt.plot(tt, cos4(tt, *popt), 'orange', label='Fit cos^4')
plt.xlabel("alpha (°)")
plt.ylabel("S21 normalisé")
plt.title(r'$\theta_E$ = {:.3f} ± {:.3f}°'.format(popt[0], error[0]))
plt.grid(10)
plt.tight_layout()
#fig.savefig(f"../Plots/Scan_Fit_mean.png", dpi=300)

In [ ]:
def cos2cos2(alpha, delta_theta, amp=1, offset1=0, offset2=0):
    return amp * (np.cos(np.radians(alpha))**2+ offset1) * (np.cos(np.radians(alpha + 90 - delta_theta))**2+ offset2)

# Sans donner d'erreur au fit
# Take the mean
t = 1

signal = mean_over_freqscans[:, t]

# Fit the data
popt, pcov = curve_fit(cos2cos2, alpha, signal, absolute_sigma=False)
error = np.sqrt(np.diag(pcov))

print("Paramètres optimisés :", popt)
print("Erreurs sur les paramètres :", error)

# Residus et chi2
residuals = signal - cos2cos2(alpha, *popt)
std_res = np.std(residuals)
print("Écart-type des résidus :", std_res)
chi2 = get_chi2(signal, cos2cos2(alpha, *popt), std_res)
print("Chi2 :", chi2)

chi2_red = chi2 / (len(alpha) - len(popt))
print("Chi2 réduit :", chi2_red)

plt.figure()
plt.errorbar(alpha, signal, marker='.', ls='None')
plt.plot(tt, cos2cos2(tt, *popt), 'orange', label='Fit cos^4')
plt.xlabel("alpha (°)")
plt.ylabel("S21 normalisé")
plt.title(r'$\theta_E$ = {:.3f} ± {:.3f}°'.format(popt[0], error[0]))
plt.grid(10)

# Residus 

On soustrait le modèle calculé sur la moyenne à chaque fréquence pour essayer d'identifier une systématique.

In [ ]:
the_model = cos4(alpha, *popt)

In [ ]:
residus_freq = mag_lin - the_model[None, :, None, None]
print(residus_freq.shape)

In [ ]:
### Plot chaque fréquence
s = 0 # Scan

freqs = [0, 2, 3, 10, 15, 20, 30, 40]
nf = len(freqs)
tt = np.arange(0, 380)
colors, norm, cmap = get_colors(nf, cmap_name='viridis', vmin=0, vmax=nf-1)
fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax = ax.ravel()
for i, f in enumerate(freqs):
    ax[0].plot(alpha, residus_freq[s, :, 0, f], 'o', color=colors[i]) # S21
    ax[1].plot(alpha, residus_freq[s, :, 1, f], 'o', color=colors[i]) # S12

ax[0].set_xlabel("alpha (°)")
ax[0].set_ylabel("S21 - fit")
#ax[0].set_yscale('log')
ax[0].grid(10)

ax[1].set_xlabel("alpha (°)")
ax[1].set_ylabel("S12 - fit")
#ax[1].set_yscale('log')
ax[1].grid(10)
# créer une colorbar qui montre l'échelle d'indices (ou de valeurs réelles)
sm = ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])  # nécessaire pour colorbar
cbar = plt.colorbar(sm, ax=ax[0])
cbar = plt.colorbar(sm, ax=ax[1])
cbar.set_label('Indice de la fréquence')

fig.tight_layout()